# 01 · OSM gazetteer for Hyderabad (CPU)

**What this does:** downloads OpenStreetMap (OSM) data, cuts out the Hyderabad box, and saves clean tables that every later notebook builds on.

| Output table | What is in it |
|---|---|
| `buildings.parquet` | One row per building: a point inside it, footprint area, H3 cells (res 7–11), DIGIPIN, nearest road access point |
| `landmarks.parquet` | Named places (temples, schools, malls, metro stations, hospitals, shops...) with a category |
| `roads.parquet` | Road lines with type and name |
| `addr_tagged.parquet` | Features that already carry real address tags (`addr:*`) |
| `entrances.parquet` | Entrance and gate nodes |
| `areas.parquet` | Named residential complexes, neighbourhoods, and any postcode boundaries |
| `landmark_areas.parquet` | Large named places drawn as shapes (campuses, malls, parks), for "inside X" addresses |

**Runtime:** Colab, CPU only (Runtime → Change runtime type → CPU). About 20–40 minutes, most of it the download.

**Where files go:** your Google Drive, `MyDrive/Navigate/data/`. The big download is cached, so re-running is fast.

The last cell prints a **coverage report**: keep it, later notebooks are judged against it.

## 1. Settings and Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
BBOX = dict(west=78.25, south=17.25, east=78.70, north=17.60)   # Hyderabad study area
CITY = 'hyderabad'

ROOT    = '/content/drive/MyDrive/Navigate/data'
RAW     = f'{ROOT}/raw/osm'
OUT     = f'{ROOT}/gazetteer/{CITY}'
WORK    = '/content/work'            # fast local disk for temporary files
for p in (RAW, OUT, WORK): os.makedirs(p, exist_ok=True)

# Geofabrik splits India into zones; Telangana is in the Southern Zone (~0.5 GB).
PBF_URL      = 'https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf'
PBF_FALLBACK = 'https://download.geofabrik.de/asia/india-latest.osm.pbf'   # whole India, ~1.5 GB
PBF_PATH     = f'{RAW}/southern-zone-latest.osm.pbf'
print('Outputs will go to', OUT)

## 2. Install tools
`osmium` is the standard command-line tool for cutting and filtering OSM files. `h3` is Uber's hexagon grid.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y osmium-tool > /dev/null
!pip -q install h3
!osmium --version | head -1
import h3, geopandas as gpd, pandas as pd, numpy as np, shapely
print('h3', h3.__version__, '| geopandas', gpd.__version__, '| shapely', shapely.__version__)

## 3. Download the OSM extract (cached on Drive)

In [ ]:
import subprocess
def sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError('command failed')
    return r.stdout

if not os.path.exists(PBF_PATH) or os.path.getsize(PBF_PATH) < 50_000_000:
    sh(f'wget -q -c -O "{PBF_PATH}" {PBF_URL}')
print(f'{os.path.getsize(PBF_PATH)/1e6:.0f} MB downloaded')
print(sh(f'osmium fileinfo -g header.option.osmosis_replication_timestamp "{PBF_PATH}"').strip() or '(no timestamp in header)')

## 4. Cut out the Hyderabad box, then split into themed files

In [ ]:
b = BBOX
CITY_PBF = f'{WORK}/{CITY}.osm.pbf'
sh(f'osmium extract -b {b["west"]},{b["south"]},{b["east"]},{b["north"]} "{PBF_PATH}" -o {CITY_PBF} --overwrite')

# Each theme = an osmium tag filter. nwr = nodes, ways and relations.
THEMES = {
    'buildings': 'w/building r/building',
    'landmarks': ('nwr/amenity nwr/shop nwr/tourism nwr/leisure nwr/office nwr/historic nwr/healthcare '
                  'nwr/craft nwr/sport nwr/railway=station,halt,tram_stop,subway_entrance '
                  'nwr/public_transport=station,stop_position,platform nwr/highway=bus_stop '
                  'nwr/man_made=water_tower,tower nwr/building=temple,church,mosque,hospital,school,college,university,commercial,retail,office'),
    'roads':     'w/highway',
    'addr':      'nwr/addr:housenumber nwr/addr:street nwr/addr:postcode nwr/addr:full nwr/addr:city',
    'entrances': 'n/entrance n/barrier=gate,lift_gate,entrance',
    'areas':     'nwr/landuse=residential nwr/place=neighbourhood,suburb,quarter,locality,village,hamlet nwr/boundary=postal_code',
}
for name, expr in THEMES.items():
    sh(f'osmium tags-filter {CITY_PBF} {expr} -o {WORK}/{name}.osm.pbf --overwrite')
    sh(f'osmium export {WORK}/{name}.osm.pbf -f geojsonseq -x print_record_separator=false '
       f'-a type,id -o {WORK}/{name}.geojsonseq --overwrite')
    print(f'  {name}: {os.path.getsize(f"{WORK}/{name}.geojsonseq")/1e6:.1f} MB')

## 5. Load into tables

In [ ]:
from shapely.geometry import shape, box
AOI = box(BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north'])

def load(name, keep=None, prefer='area'):
    rows, geoms = [], []
    with open(f'{WORK}/{name}.geojsonseq') as f:
        for line in f:
            line = line.strip().lstrip('\x1e')
            if not line: continue
            ft = json.loads(line)
            p = ft['properties']
            p['osm_id'] = f"{p.pop('@type','?')[0]}{p.pop('@id','')}"   # e.g. w123456
            rows.append(p if keep is None else {k: p.get(k) for k in ['osm_id', *keep]})
            geoms.append(shape(ft['geometry']))
    g = gpd.GeoDataFrame(rows, geometry=geoms, crs='EPSG:4326')
    g = g[g.intersects(AOI)].reset_index(drop=True)          # osmium keeps whole ways that cross the edge
    # A closed OSM way (a loop) is exported twice: once as a line and once as an area. Keep one row per feature.
    is_area = g.geom_type.isin(['Polygon','MultiPolygon'])
    g = g.assign(_rank=is_area if prefer == 'area' else ~is_area).sort_values('_rank', ascending=False)
    g = g.drop_duplicates('osm_id').drop(columns='_rank').sort_index().reset_index(drop=True)
    print(f'{name:10s} {len(g):>9,} features')
    return g

ADDR_KEYS = ['addr:housenumber','addr:housename','addr:flats','addr:unit','addr:street','addr:place',
             'addr:suburb','addr:city','addr:postcode','addr:full']

buildings = load('buildings', keep=['building','name','building:levels', *ADDR_KEYS])
buildings = buildings[buildings.geom_type.isin(['Polygon','MultiPolygon'])].reset_index(drop=True)
landmarks_raw = load('landmarks')
roads     = load('roads', keep=['highway','name','name:te','name:hi','ref','oneway','junction'], prefer='line')
roads     = roads[roads.geom_type.isin(['LineString','MultiLineString'])].reset_index(drop=True)
addr      = load('addr', keep=['name', *ADDR_KEYS])
entrances = load('entrances', keep=['entrance','barrier','name','access'])
areas     = load('areas', keep=['name','name:te','name:hi','landuse','place','boundary','postal_code','addr:postcode'])

## 6. Clean landmarks: keep named ones and give each a category

In [ ]:
CAT_KEYS = ['amenity','shop','tourism','leisure','office','historic','healthcare','craft','sport',
            'railway','public_transport','highway','man_made','building']
lm = landmarks_raw.copy()
for k in CAT_KEYS + ['name','name:en','name:te','name:hi','brand']:
    if k not in lm: lm[k] = None
lm['name'] = lm['name'].fillna(lm['name:en']).fillna(lm['brand'])
lm = lm[lm['name'].notna()].copy()

def category(row):
    for k in CAT_KEYS:
        v = row.get(k)
        if isinstance(v, str) and v not in ('yes', 'no'): return f'{k}={v}'
    return 'other'
lm['category'] = lm.apply(category, axis=1)
# Big named places drawn as areas (campuses, malls, parks, hospitals): kept as shapes too, so a later notebook
# can tell when a building is INSIDE one of them.
big = lm[lm.geom_type.isin(['Polygon','MultiPolygon'])].copy()
big['area_m2'] = big.to_crs('EPSG:32644').area.round(0)
landmark_areas = big[(big.area_m2 >= 2000) & (big.area_m2 <= 1.5e6)][['osm_id','name','category','area_m2','geometry']].reset_index(drop=True)
print(f'{len(landmark_areas):,} named landmark areas of 2,000 sq m or more')

lm['geometry'] = lm.geometry.representative_point()        # one point per landmark
landmarks = lm[['osm_id','name','name:te','name:hi','category', *[k for k in ADDR_KEYS if k in lm], 'geometry']].reset_index(drop=True)
landmarks = gpd.GeoDataFrame(landmarks, crs='EPSG:4326')
print(f'{len(landmarks):,} named landmarks')
print(landmarks['category'].value_counts().head(25).to_string())

## 7. Per-building columns: point, area, H3 cells, DIGIPIN, road access point
- **Point:** a point guaranteed to sit inside the footprint (`representative_point`), used as ground truth.
- **DIGIPIN:** India Post's 10-character code for a ~4 m cell. The encoder below is checked against India Post's published example.
- **Road access point:** the nearest spot on a drivable road. Used later for 'across the road' and pickup-style metrics.

In [ ]:
UTM = 'EPSG:32644'     # UTM zone 44N, metres, covers Hyderabad

b = buildings.copy()
b['area_m2'] = b.to_crs(UTM).area.round(1)
pts = b.geometry.representative_point()
b['lat'], b['lon'] = pts.y.values, pts.x.values

for res in (7, 8, 9, 10, 11):
    b[f'h3_{res}'] = [h3.latlng_to_cell(la, lo, res) for la, lo in zip(b.lat, b.lon)]

DIGI = [['F','C','9','8'],['J','3','2','7'],['K','4','5','6'],['L','M','P','T']]
def digipin(lat, lon):
    minlat, maxlat, minlon, maxlon = 2.5, 38.5, 63.5, 99.5
    out = ''
    for _ in range(10):
        dlat, dlon = (maxlat-minlat)/4, (maxlon-minlon)/4
        r = min(max(3 - int((lat-minlat)//dlat), 0), 3)
        c = min(max(int((lon-minlon)//dlon), 0), 3)
        out += DIGI[r][c]
        maxlat = minlat + dlat*(4-r); minlat = maxlat - dlat
        minlon = minlon + dlon*c;     maxlon = minlon + dlon
    return out
assert digipin(28.622788, 77.213033) == '39J49LL8T4', 'DIGIPIN encoder check failed'   # Dak Bhawan, India Post's example
b['digipin'] = [digipin(la, lo) for la, lo in zip(b.lat, b.lon)]

DRIVABLE = {'motorway','trunk','primary','secondary','tertiary','unclassified','residential','service',
            'living_street','road','motorway_link','trunk_link','primary_link','secondary_link','tertiary_link'}
drive_roads = roads[roads.highway.isin(DRIVABLE)].to_crs(UTM).reset_index(drop=True)
bp = gpd.GeoSeries(gpd.points_from_xy(b.lon, b.lat), crs='EPSG:4326').to_crs(UTM)

tree = shapely.STRtree(drive_roads.geometry.values)
(src, dst), dist = tree.query_nearest(bp.values, return_distance=True, all_matches=False)
order = np.argsort(src); src, dst, dist = src[order], dst[order], dist[order]
lines = drive_roads.geometry.values[dst]
snap  = shapely.line_interpolate_point(lines, shapely.line_locate_point(lines, bp.values[src]))
snap_ll = gpd.GeoSeries(snap, crs=UTM).to_crs('EPSG:4326')
b['access_lat'], b['access_lon'] = np.nan, np.nan
b.loc[src, 'access_lat'] = snap_ll.y.values
b.loc[src, 'access_lon'] = snap_ll.x.values
b.loc[src, 'access_road_osm_id'] = drive_roads.osm_id.values[dst]
b.loc[src, 'access_road_name']   = drive_roads['name'].values[dst]
b.loc[src, 'dist_to_road_m']     = dist.round(1)
print(b[['lat','lon','h3_10','digipin','access_lat','access_lon','dist_to_road_m']].head())

## 8. Coverage report
The key question: do enough buildings have a **named landmark nearby** to write realistic 'near X / opposite Y' addresses?

In [ ]:
lm_utm = landmarks.to_crs(UTM)
lt = shapely.STRtree(lm_utm.geometry.values)
(bi, li), ld = lt.query_nearest(bp.values, max_distance=300, return_distance=True, all_matches=False)
b['nearest_landmark_osm_id'] = None; b['nearest_landmark_m'] = np.nan
b.loc[bi, 'nearest_landmark_osm_id'] = lm_utm.osm_id.values[li]
b.loc[bi, 'nearest_landmark_m'] = ld.round(1)

# landmarks within 300 m, measured on a sample to keep memory low
samp = bp.sample(min(20000, len(bp)), random_state=0)
pairs = lt.query(samp.values, predicate='dwithin', distance=300)
per_bldg = np.bincount(pairs[0], minlength=len(samp))

named_complex = areas[(areas.get('landuse') == 'residential') & areas['name'].notna() & areas.geom_type.isin(['Polygon','MultiPolygon'])]
report = {
    'buildings': len(b),
    'buildings_with_name': int(b['name'].notna().sum()),
    'buildings_with_any_addr_tag': int(b[[k for k in ADDR_KEYS if k in b]].notna().any(axis=1).sum()),
    'named_landmarks': len(landmarks),
    'named_landmark_areas': len(landmark_areas),
    'roads_all': len(roads),
    'roads_drivable': len(drive_roads),
    'roads_named_share': round(float(roads['name'].notna().mean()), 3),
    'addr_tagged_features': len(addr),
    'entrance_nodes': len(entrances),
    'named_residential_complexes': len(named_complex),
    'postcode_boundaries': int((areas.get('boundary') == 'postal_code').sum()),
    'share_bldg_landmark_within_100m': round(float((b.nearest_landmark_m <= 100).mean()), 3),
    'share_bldg_landmark_within_300m': round(float(b.nearest_landmark_m.notna().mean()), 3),
    'median_landmarks_within_300m': float(np.median(per_bldg)),
    'median_dist_to_drivable_road_m': float(b.dist_to_road_m.median()),
}
print(json.dumps(report, indent=2))

## 9. Quick map (sanity check)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].hexbin(b.lon, b.lat, gridsize=80, bins='log', cmap='Greys'); ax[0].set_title('Buildings (log density)')
ax[1].hexbin(landmarks.geometry.x, landmarks.geometry.y, gridsize=80, bins='log', cmap='Greys'); ax[1].set_title('Named landmarks (log density)')
for a in ax: a.set_aspect('equal'); a.set_xlim(BBOX['west'], BBOX['east']); a.set_ylim(BBOX['south'], BBOX['north'])
plt.tight_layout(); plt.savefig(f'{OUT}/coverage_map.png', dpi=110); plt.show()

## 10. Save

In [ ]:
def save(g, name):
    g = g.copy()
    for c in g.columns:                      # mixed-type tag columns -> strings, so parquet is happy
        if c != 'geometry' and g[c].dtype == object: g[c] = g[c].astype('string')
    g.to_parquet(f'{OUT}/{name}.parquet', index=False)
    print(f'{name:12s} {len(g):>9,} rows  {os.path.getsize(f"{OUT}/{name}.parquet")/1e6:6.1f} MB')

save(gpd.GeoDataFrame(b, geometry='geometry', crs='EPSG:4326'), 'buildings')
save(landmarks, 'landmarks'); save(roads, 'roads'); save(addr, 'addr_tagged')
save(entrances, 'entrances'); save(areas, 'areas'); save(landmark_areas, 'landmark_areas')

manifest = {'city': CITY, 'bbox': BBOX, 'source': PBF_URL, 'created': time.strftime('%Y-%m-%d %H:%M'),
            'licence': 'OpenStreetMap contributors, ODbL 1.0', 'report': report}
json.dump(manifest, open(f'{OUT}/manifest.json', 'w'), indent=2)
print('Done.')